# Small Molecules

This notebook shows embpy's molecule workflow: validate and canonicalize SMILES, embed compounds with fingerprints or chemical language models, annotate chemistry, and compare molecular embedding spaces.


## embpy capability map

All embpy tutorials follow the same package pattern:

1. `BioEmbedder.embed(...)` is the single entry point for genes, proteins, molecules, perturbation images, text, and single cells.
2. Resolvers convert biological identifiers into model-ready inputs, such as gene sequences, protein sequences, SMILES strings, microscopy tensors, or AnnData matrices.
3. The model registry selects the requested embedding backend, from lightweight local baselines to foundation models.
4. Preprocessing utilities in `embpy.pp` prepare inputs when a model needs a specific representation, such as raw counts, log-normalized expression, or morphology canvases.
5. Metadata tools in `embpy.tl` and `embpy.resources` annotate the resulting AnnData with genes, proteins, molecules, perturbations, and cell-line metadata.
6. Plotting and comparison helpers in `embpy.pl` and `embpy.tl` inspect embedding geometry, cluster structure, KNN overlap, similarity, and annotation enrichment.

The important contract is that embeddings are stored in AnnData-friendly locations: row-aligned embeddings in `.obsm`, feature-aligned embeddings in `.varm`, and entity-aligned payloads or provenance in `.uns`. `.X` stays reserved for count/expression-like data or a lightweight placeholder.


## What this notebook demonstrates

- Molecules can enter as SMILES, names, or tables and are normalized before embedding.
- Lightweight fingerprints such as Morgan fingerprints can be mixed with neural molecule models such as ChemBERTa and MolFormer.
- Molecule annotations add structural properties, bioactivity, ontology, pathway, and cross-reference metadata when available.
- Embedding plots and similarity diagnostics help compare whether different molecule models organize compounds similarly.


In [ ]:
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
from IPython.display import display
from rdkit import Chem

from embpy import BioEmbedder, pl, tl

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

embedder = BioEmbedder(device="auto", organism="human")


def compact_obs(adata: ad.AnnData, prefixes: tuple[str, ...], base: list[str] | None = None) -> pd.DataFrame:
    base = base or []
    cols = [c for c in base if c in adata.obs.columns]
    cols += [c for c in adata.obs.columns if c.startswith(prefixes)]
    return adata.obs.loc[:, list(dict.fromkeys(cols))]


def canonicalize_smiles(value: str) -> str:
    mol = Chem.MolFromSmiles(value)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {value}")
    return Chem.MolToSmiles(mol)


compounds = ["aspirin", "ibuprofen", "caffeine", "acetaminophen", "imatinib", "gefitinib"]
smiles = [
    "CC(=O)OC1=CC=CC=C1C(=O)O",
    "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O",
    "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
    "CC(=O)NC1=CC=C(O)C=C1",
    "CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C)NC4=NC=CC(=N4)C5=CN=CC=C5",
    "COC1=C(C=C2C(=C1)N=CN=C2NC3=CC(=C(C=C3)F)Cl)OCCCN4CCOCC4",
]
molecule_table = pd.DataFrame(
    {
        "compound": compounds,
        "smiles": smiles,
        "canonical_smiles": [canonicalize_smiles(s) for s in smiles],
    }
)

molecule_space = ad.AnnData(
    X=np.zeros((len(molecule_table), 1), dtype=np.float32),
    obs=molecule_table.set_index("canonical_smiles", drop=False).copy(),
    var=pd.DataFrame(index=["placeholder_feature"]),
)

display(molecule_table)
display(molecule_space)


## Embed molecules

The tutorial uses a small AnnData with fake `.X`, real compound labels, and real SMILES. The Morgan fingerprint is computed locally with RDKit; ChemBERTa and MolFormer use the registered model wrappers.


In [ ]:
for model_name, key in [
    ("morgan_fp", "X_morgan_fp"),
    ("chemberta2MTR", "X_chemberta2MTR"),
    ("molformer_base", "X_molformer_base"),
]:
    molecule_space = embedder.embed(
        molecule_space,
        entity_type="molecule",
        id_type="smiles",
        obs_column="smiles",
        model=model_name,
        output="anndata",
        attach_to="obs",
        key=key,
        show_progress=True,
    )

if "smiles" not in molecule_space.obs:
    molecule_space.obs["smiles"] = molecule_space.obs_names.astype(str)
if "compound" not in molecule_space.obs:
    molecule_space.obs["compound"] = compounds[: molecule_space.n_obs]

display(molecule_space)
print("obsm keys for plotting:", list(molecule_space.obsm.keys()))


## Annotate molecules


In [ ]:
molecule_space = tl.annotate_molecules(
    molecule_space,
    column="smiles",
    sources=["structural", "bioactivity", "ontology", "pathways", "xrefs"],
    copy=True,
)

display(compact_obs(molecule_space, ("mol_",), base=["compound", "smiles"]))
print("annotation stores:", [k for k in molecule_space.uns if "annotation" in k])


## Plot annotated molecule spaces


In [ ]:
color_key = "mol_logp" if "mol_logp" in molecule_space.obs else "compound"
pl.plot_embedding_space(
    molecule_space,
    obsm_key="X_morgan_fp",
    method="pca",
    color=color_key,
    annotate=True,
    annotate_col="compound",
    title="Morgan fingerprints colored by embpy molecule annotations",
)

if "mol_qed" in molecule_space.obs:
    pl.plot_embedding_space(
        molecule_space,
        obsm_key="X_chemberta2MTR",
        method="pca",
        color="mol_qed",
        annotate=True,
        annotate_col="compound",
        title="ChemBERTa embeddings colored by QED",
    )


## Compare molecule models


In [ ]:
k = min(2, molecule_space.n_obs - 1)
_, mean_overlap = tl.compute_knn_overlap(molecule_space, "X_morgan_fp", "X_chemberta2MTR", k=k)
print(f"Mean Morgan/ChemBERTa KNN overlap: {mean_overlap:.3f}")

pl.knn_overlap(molecule_space, obsm_keys=["X_morgan_fp", "X_chemberta2MTR", "X_molformer_base"], k=k)
pl.cross_embedding_correlation(molecule_space, "X_morgan_fp", "X_molformer_base")
pl.embedding_norms(molecule_space, obsm_keys=["X_morgan_fp", "X_chemberta2MTR", "X_molformer_base"])


## Summary

This notebook covered the small-molecule embpy pattern: canonicalize molecular identifiers, embed molecules with both cheminformatics and foundation-model representations, annotate compounds, compare embedding views, and write the result as AnnData.


## Save a reusable artifact


In [ ]:
molecule_space.write_h5ad(OUTPUT_DIR / "molecule_embeddings.h5ad")
print(OUTPUT_DIR / "molecule_embeddings.h5ad")
